# Figure: Surface vapor composition

In [ ]:
from pathlib import Path
import numpy as np

results_directory = Path().resolve().parent / "Model_Outputs"
SAVE_FIG = True

## Import data and styling

In [ ]:
import matplotlib.pyplot as plt
from cycler import cycler
from helpers.degassing_data import load_all_systems

# --- USER INPUTS --- #
SAMPLES = ["MORB", "Kilauea", "Fuego", "Fogo"]
TOOLS  = ["DCompress", "DCompress (IM)", "EVo", "MAGEC", "SulfurX", "VolFe",]

systems = load_all_systems(SAMPLES, TOOLS, results_dir=results_directory)

# --- PLOT COLOR STYLING --- #
colors = ["#404040", "#bababa", "#dfe6e9", "#2c7bb6", "#abd9e9", "#bd0026", "#fc7d22", "#ffbe32", "#54278f",]
plt.rcParams['axes.prop_cycle'] = cycler(color=colors)

## Plot

In [ ]:
SPECIES = ["O2_v_mf","CO2_v_mf","CO_v_mf","H2O_v_mf","H2_v_mf","S2_v_mf","SO2_v_mf","H2S_v_mf","CH4_v_mf"]
SPECIES_n = ["O$_2$","CO$_2$","CO","H$_2$O","H$_2$","S$_2$","SO$_2$","H$_2$S","CH$_4$"]

Mwt = np.array([32e-3, 44e-3, 28e-3, 18.01e-3, 2e-3, 64e-3, 64e-3, 34e-3, 16e-3])

fig, ax = plt.subplots(ncols=4, sharey=True, dpi=130)

for c, sample in enumerate(SAMPLES, start=1):

    VapArr = np.zeros([len(TOOLS),len(SPECIES)])
    VapArr_w = np.zeros([len(TOOLS),len(SPECIES)])
    
    i = int(0)
    for tool in TOOLS:
        df = systems.get(sample, {}).get(tool)
        if df is None or "P_bars" not in df.columns:
            continue

        SumM = np.zeros([len(SPECIES)])
        j = int(0)
        for species in SPECIES :
            if np.isfinite(df[species].iloc[-1]) :
                VapArr[i,j] = df[species].iloc[-1]
            j += 1
        # convert VapArr form mol fr. to wt fr
        SumM[:] = VapArr[i,:]*Mwt[:] 
        VapArr_w[i,:] = SumM[:]/SumM.sum()
        
        i += 1
   
    Vap = {SPECIES_n[1]: VapArr[:,1], SPECIES_n[2]: VapArr[:,2], SPECIES_n[8]: VapArr[:,8], SPECIES_n[3]: VapArr[:,3],
           SPECIES_n[4]: VapArr[:,4], SPECIES_n[6]: VapArr[:,6], SPECIES_n[7]: VapArr[:,7], SPECIES_n[5]: VapArr[:,5],
           SPECIES_n[0]: VapArr[:,0] } 
    
    Vap_w = {SPECIES_n[1]: VapArr_w[:,1], SPECIES_n[2]: VapArr_w[:,2], SPECIES_n[8]: VapArr_w[:,8], SPECIES_n[3]: VapArr_w[:,3],
           SPECIES_n[4]: VapArr_w[:,4], SPECIES_n[6]: VapArr_w[:,6], SPECIES_n[7]: VapArr_w[:,7], SPECIES_n[5]: VapArr_w[:,5],
           SPECIES_n[0]: VapArr_w[:,0] } 

    width = 0.6

    bottom = np.zeros(len(TOOLS))
    for i,j in Vap_w.items():
        p = ax[c-1].bar(TOOLS, j, width, label=i, bottom=bottom)
        #plt.xticks(rotation=90)
        ax[c-1].tick_params(axis='x', labelrotation=90)
        bottom += j
    
    ax[c-1].set_title(sample)

handles, labels = ax[3].get_legend_handles_labels()
ax[3].legend(handles[::-1], labels[::-1], bbox_to_anchor=(1.2, 0.8),
             frameon=True, edgecolor='black', fancybox=False)

if SAVE_FIG:
    fig.savefig("figures/Fig_vapor_surf.png", dpi=300, bbox_inches='tight')

plt.show()

